# Work Notebook

## Phase 1. CS Corpus Acquisition and OpenAlex-Gated 50K Selection

- **Goal:** Build a reproducible candidate CS pool from arXiv, then produce a final 50K corpus with valid institution metadata.
- **Tasks:** Load metadata, validate required fields, filter `cs.*`, remove missing title/abstract and duplicate IDs, build a candidate pool, then gate selection with batched OpenAlex DOI lookups.
- **Expected outputs:** `raw_data_profile.json`, `cs_candidate_pool.parquet`, `final_50k_with_institution.parquet`, `openalex_gate_cache.json`, `openalex_gate_report.json`, `corpus_selection_report.json`.
- **Optional output (audit mode):** `eligible_cs_pool.parquet` (generated only when `SAVE_ELIGIBLE_POOL=True`).

In [1]:
import json
import os
import re
import time
from datetime import datetime, timezone

import duckdb
import kagglehub
import pandas as pd
import requests

# --------------------------
# Reproducibility / outputs
# --------------------------
DATA_DIR = "data"
SAVE_ELIGIBLE_POOL = False  # Set True only when you need an auditable eligible pool file
TARGET_VALID_SIZE = 50000  # For quick validation: set 1000, then 10000, then 50000
SAMPLE_SEED = 42

OPENALEX_BATCH_SIZE = 50
OPENALEX_TIMEOUT_SECONDS = 30
OPENALEX_MAX_RETRIES = 5
OPENALEX_BASE_SLEEP_SECONDS = 0.4
OPENALEX_INTER_BATCH_SLEEP_SECONDS = 0.1

REQUIRED_FIELDS = [
    "id",
    "title",
    "abstract",
    "authors",
    "categories",
    "versions",
    "update_date",
]

os.makedirs(DATA_DIR, exist_ok=True)

raw_profile_path = os.path.join(DATA_DIR, "raw_data_profile.json")
eligible_pool_path = os.path.join(DATA_DIR, "eligible_cs_pool.parquet")
candidate_pool_path = os.path.join(DATA_DIR, "cs_candidate_pool.parquet")
final_valid_corpus_path = os.path.join(DATA_DIR, "final_50k_with_institution.parquet")
openalex_gate_cache_path = os.path.join(DATA_DIR, "openalex_gate_cache.json")
openalex_gate_report_path = os.path.join(DATA_DIR, "openalex_gate_report.json")
selection_report_path = os.path.join(DATA_DIR, "corpus_selection_report.json")

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
openalex_config_path = os.path.join(project_root, "openalex api.txt")

# --------------------------
# Download fixed source snapshot
# --------------------------
DATASET_OWNER = "Cornell-University"
DATASET_NAME = "arxiv"
DATASET_VERSION = 289

dataset_source = f"{DATASET_OWNER}/{DATASET_NAME}/versions/{DATASET_VERSION}"
dataset_path = kagglehub.dataset_download(dataset_source)
json_file_path = os.path.join(dataset_path, "arxiv-metadata-oai-snapshot.json")

print("Dataset source (fixed):", dataset_source)
print("Dataset path:", dataset_path)
print("JSON path:", json_file_path)
print("Outputs directory:", DATA_DIR)
print("OpenAlex config path:", openalex_config_path)

c:\Users\Darjeel1ng\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset source (fixed): Cornell-University/arxiv/versions/289
Dataset path: C:\Users\Darjeel1ng\.cache\kagglehub\datasets\Cornell-University\arxiv\versions\289
JSON path: C:\Users\Darjeel1ng\.cache\kagglehub\datasets\Cornell-University\arxiv\versions\289\arxiv-metadata-oai-snapshot.json
Outputs directory: data
OpenAlex config path: c:\Users\Darjeel1ng\OneDrive - Northeastern University\Attachments\cs 6200\project\openalex api.txt


In [2]:
# --------------------------
# Validate required fields + raw profile
# --------------------------
schema_df = duckdb.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_json_auto('{json_file_path}', format='newline_delimited')
    LIMIT 1;
    """
).df()

available_fields = set(schema_df["column_name"].tolist())
missing_required_fields = [f for f in REQUIRED_FIELDS if f not in available_fields]

if missing_required_fields:
    raise ValueError(f"Missing required fields: {missing_required_fields}")

print("Required fields check passed.")

raw_profile_query = f"""
SELECT
    count(1) AS total_records,
    count_if(categories LIKE 'cs.%' OR categories LIKE '% cs.%') AS cs_records,
    count_if(NOT (categories LIKE 'cs.%' OR categories LIKE '% cs.%')) AS non_cs_records,
    count_if(title IS NULL OR TRIM(title) = '') AS missing_title,
    count_if(abstract IS NULL OR TRIM(abstract) = '') AS missing_abstract,
    count_if(authors IS NULL OR TRIM(CAST(authors AS VARCHAR)) = '') AS missing_authors,
    count_if(versions IS NULL OR CAST(versions AS VARCHAR) = '[]') AS missing_versions,
    count_if(update_date IS NULL) AS missing_update_date,
    min(update_date) AS earliest_update,
    max(update_date) AS latest_update
FROM read_json_auto('{json_file_path}', format='newline_delimited');
"""

raw_profile = duckdb.sql(raw_profile_query).df().iloc[0].to_dict()

raw_profile_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_source": dataset_source,
    "dataset_path": dataset_path,
    "json_file_path": json_file_path,
    "required_fields": REQUIRED_FIELDS,
    "required_fields_check_passed": True,
    "profile": {k: (v if isinstance(v, (int, float)) else str(v)) for k, v in raw_profile.items()},
}

with open(raw_profile_path, "w", encoding="utf-8") as f:
    json.dump(raw_profile_report, f, indent=2)

print("raw_data_profile written:", raw_profile_path)
print("Raw profile summary:")
for k, v in raw_profile.items():
    print(f"  {k}: {v}")

Required fields check passed.
raw_data_profile written: data\raw_data_profile.json
Raw profile summary:
  total_records: 3066190
  cs_records: 955380.0
  non_cs_records: 2110810.0
  missing_title: 0.0
  missing_abstract: 0.0
  missing_authors: 0.0
  missing_versions: 0.0
  missing_update_date: 0.0
  earliest_update: 2007-05-23 00:00:00
  latest_update: 2026-06-05 00:00:00


In [3]:
# --------------------------
# Selection stats (correct accounting)
# --------------------------
selection_stats_query = f"""
WITH source AS (
    SELECT *
    FROM read_json_auto('{json_file_path}', format='newline_delimited')
),
cs_only AS (
    SELECT *, regexp_replace(id, 'v[0-9]+$', '') AS canonical_id
    FROM source
    WHERE categories LIKE 'cs.%' OR categories LIKE '% cs.%'
),
non_empty_text AS (
    SELECT *
    FROM cs_only
    WHERE title IS NOT NULL AND TRIM(title) != ''
      AND abstract IS NOT NULL AND TRIM(abstract) != ''
),
deduped AS (
    SELECT *
    FROM non_empty_text
    QUALIFY ROW_NUMBER() OVER (PARTITION BY canonical_id ORDER BY update_date DESC) = 1
)
SELECT
    (SELECT count(1) FROM source) AS total_records,
    (SELECT count(1) FROM cs_only) AS cs_records,
    (SELECT count_if(id ~ '.*v[0-9]+$') FROM cs_only) AS cs_ids_with_version_suffix,
    (SELECT count(1) FROM cs_only WHERE title IS NULL OR TRIM(title) = '' OR abstract IS NULL OR TRIM(abstract) = '') AS dropped_missing_fields,
    (SELECT count(1) FROM non_empty_text) AS cs_with_non_empty_text,
    (SELECT count(1) FROM deduped) AS eligible_pool_size,
    (SELECT count(1) FROM non_empty_text) - (SELECT count(1) FROM deduped) AS dropped_duplicates
"""

selection_stats = duckdb.sql(selection_stats_query).df().iloc[0].to_dict()
eligible_pool_size = int(selection_stats["eligible_pool_size"])

print("Selection stats:")
for k, v in selection_stats.items():
    print(f"  {k}: {v}")

if SAVE_ELIGIBLE_POOL:
    eligible_pool_query = f"""
    COPY (
        SELECT *
        FROM (
            SELECT
                *,
                regexp_replace(id, 'v[0-9]+$', '') AS canonical_id
            FROM read_json_auto('{json_file_path}', format='newline_delimited')
            WHERE categories LIKE 'cs.%' OR categories LIKE '% cs.%'
              AND title IS NOT NULL AND TRIM(title) != ''
              AND abstract IS NOT NULL AND TRIM(abstract) != ''
        )
        QUALIFY ROW_NUMBER() OVER (PARTITION BY canonical_id ORDER BY update_date DESC) = 1
    )
    TO '{eligible_pool_path}' (FORMAT PARQUET);
    """
    duckdb.sql(eligible_pool_query)
    print("eligible_cs_pool written:", eligible_pool_path)
else:
    print("eligible_cs_pool skipped (SAVE_ELIGIBLE_POOL=False)")

Selection stats:
  total_records: 3066190.0
  cs_records: 955380.0
  cs_ids_with_version_suffix: 0.0
  dropped_missing_fields: 0.0
  cs_with_non_empty_text: 955380.0
  eligible_pool_size: 955379.0
  dropped_duplicates: 1.0
eligible_cs_pool skipped (SAVE_ELIGIBLE_POOL=False)


In [4]:
# --------------------------
# Build candidate pool, then gate with batched OpenAlex lookups
# --------------------------
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

# Run-local knobs (edit here, rerun only this cell)
OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN = None  # cids resolved via DOI this run; e.g., 1000 / 10000 / None
OPENALEX_CACHE_CHECKPOINT_EVERY_OK = 5000  # durably save cache after this many new institutions
OPENALEX_CACHE_CHECKPOINT_EVERY_BATCHES = 100  # safety floor: also save every N batches
OPENALEX_RETRY_NOT_FOUND = False  # skip papers already marked not_found (avoid re-scan)
OPENALEX_RETRY_NO_INSTITUTION = False  # skip papers already marked no_institution (avoid re-scan)
OPENALEX_USE_PUBLISHED_DOI = True  # also query the published DOI from arXiv metadata
OPENALEX_ENABLE_TITLE_SEARCH_FALLBACK = False  # recover published version via title search
OPENALEX_MAX_TITLE_SEARCHES_THIS_RUN = 2000  # cap per run; title search is one request per paper
OPENALEX_TITLE_MATCH_MIN_JACCARD = 0.82  # token-Jaccard threshold for title-search matches
OPENALEX_MAX_DOIS_PER_REQUEST = 50  # OpenAlex OR-filter value limit
CACHE_SCHEMA_VERSION = 2  # bump to re-query fails written by older strategies


def ensure_runtime_context() -> None:
    if "json_file_path" in globals():
        return

    import json
    import os
    import re
    import time
    from datetime import datetime, timezone

    import duckdb
    import kagglehub
    import pandas as pd
    import requests

    globals()["json"] = json
    globals()["os"] = os
    globals()["re"] = re
    globals()["time"] = time
    globals()["datetime"] = datetime
    globals()["timezone"] = timezone
    globals()["duckdb"] = duckdb
    globals()["kagglehub"] = kagglehub
    globals()["pd"] = pd
    globals()["requests"] = requests

    globals()["DATA_DIR"] = globals().get("DATA_DIR", "data")
    globals()["TARGET_VALID_SIZE"] = int(globals().get("TARGET_VALID_SIZE", 50000))
    globals()["OPENALEX_BATCH_SIZE"] = int(globals().get("OPENALEX_BATCH_SIZE", 50))
    globals()["OPENALEX_TIMEOUT_SECONDS"] = int(globals().get("OPENALEX_TIMEOUT_SECONDS", 30))
    globals()["OPENALEX_MAX_RETRIES"] = int(globals().get("OPENALEX_MAX_RETRIES", 5))
    globals()["OPENALEX_BASE_SLEEP_SECONDS"] = float(globals().get("OPENALEX_BASE_SLEEP_SECONDS", 0.4))
    globals()["OPENALEX_INTER_BATCH_SLEEP_SECONDS"] = float(globals().get("OPENALEX_INTER_BATCH_SLEEP_SECONDS", 0.1))

    os.makedirs(DATA_DIR, exist_ok=True)

    globals()["candidate_pool_path"] = os.path.join(DATA_DIR, "cs_candidate_pool.parquet")
    globals()["final_valid_corpus_path"] = os.path.join(DATA_DIR, "final_50k_with_institution.parquet")
    globals()["openalex_gate_cache_path"] = os.path.join(DATA_DIR, "openalex_gate_cache.json")
    globals()["openalex_gate_report_path"] = os.path.join(DATA_DIR, "openalex_gate_report.json")

    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    globals()["openalex_config_path"] = os.path.join(project_root, "openalex api.txt")

    dataset_source = "Cornell-University/arxiv/versions/289"
    dataset_path = kagglehub.dataset_download(dataset_source)
    globals()["json_file_path"] = os.path.join(dataset_path, "arxiv-metadata-oai-snapshot.json")

    print("[Gate bootstrap] Recovered missing globals after restart.")
    print("[Gate bootstrap] JSON path:", json_file_path)


ensure_runtime_context()

print("Gate run config:")
print("  OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN:", OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN)
print("  OPENALEX_CACHE_CHECKPOINT_EVERY_OK:", OPENALEX_CACHE_CHECKPOINT_EVERY_OK)
print("  OPENALEX_CACHE_CHECKPOINT_EVERY_BATCHES:", OPENALEX_CACHE_CHECKPOINT_EVERY_BATCHES)
print("  OPENALEX_RETRY_NOT_FOUND:", OPENALEX_RETRY_NOT_FOUND)
print("  OPENALEX_RETRY_NO_INSTITUTION:", OPENALEX_RETRY_NO_INSTITUTION)
print("  OPENALEX_USE_PUBLISHED_DOI:", OPENALEX_USE_PUBLISHED_DOI)
print("  OPENALEX_ENABLE_TITLE_SEARCH_FALLBACK:", OPENALEX_ENABLE_TITLE_SEARCH_FALLBACK)
print("  OPENALEX_MAX_TITLE_SEARCHES_THIS_RUN:", OPENALEX_MAX_TITLE_SEARCHES_THIS_RUN)
print("  CACHE_SCHEMA_VERSION:", CACHE_SCHEMA_VERSION)

candidate_pool_query = f"""
COPY (
    SELECT
        *,
        regexp_replace(id, 'v[0-9]+$', '') AS canonical_id
    FROM read_json_auto('{json_file_path}', format='newline_delimited')
    WHERE categories LIKE 'cs.%' OR categories LIKE '% cs.%'
      AND title IS NOT NULL AND TRIM(title) != ''
      AND abstract IS NOT NULL AND TRIM(abstract) != ''
    QUALIFY ROW_NUMBER() OVER (PARTITION BY canonical_id ORDER BY update_date DESC) = 1
)
TO '{candidate_pool_path}' (FORMAT PARQUET);
"""

duckdb.sql(candidate_pool_query)

candidate_count = int(
    duckdb.sql(f"SELECT count(1) AS count FROM read_parquet('{candidate_pool_path}')").df().iloc[0]["count"]
)

if candidate_count < TARGET_VALID_SIZE:
    raise ValueError(
        f"Candidate pool too small for target size: pool={candidate_count}, target={TARGET_VALID_SIZE}"
    )


def load_openalex_config(path: str) -> dict:
    if not os.path.exists(path):
        raise FileNotFoundError(f"OpenAlex config file not found: {path}")
    config = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            raw = line.strip()
            if not raw or raw.startswith("#") or "=" not in raw:
                continue
            k, v = raw.split("=", 1)
            config[k.strip()] = v.strip()
    missing = [k for k in ["OPENALEX_EMAIL", "OPENALEX_API_KEY"] if not config.get(k)]
    if missing:
        raise ValueError(f"Missing required OpenAlex config keys: {missing}")
    return config


def normalize_doi(raw: str) -> str:
    if not raw:
        return ""
    text = raw.strip().lower()
    text = re.sub(r"^https?://(dx\.)?doi\.org/", "", text)
    return text


DOI_RE = re.compile(r"^10\.\d{4,9}/\S+$")


def cid_to_arxiv_doi(cid: str) -> str:
    return f"10.48550/arxiv.{cid}".lower()


def is_valid_doi(raw: object) -> bool:
    return bool(DOI_RE.match(normalize_doi("" if raw is None else str(raw))))


def extract_work_institution(work: dict) -> dict:
    """Return the first usable institution (name + country) from an OpenAlex work."""
    for auth in work.get("authorships", []):
        for inst in auth.get("institutions", []):
            name = (inst.get("display_name") or "").strip()
            if name:
                country = (inst.get("country_code") or "UNK").strip().upper() or "UNK"
                return {"institution": name, "country_code": country}
    return {}


def work_is_preprint(work: dict) -> bool:
    return (work.get("type") or "").strip().lower() == "preprint"


def pick_best_work(works: list) -> tuple:
    """Among candidate works for one paper, prefer an institution-bearing, published version."""
    best = None
    best_inst = {}
    for work in works:
        inst = extract_work_institution(work)
        if not inst:
            continue
        if best is None or (work_is_preprint(best) and not work_is_preprint(work)):
            best = work
            best_inst = inst
    return best, best_inst


# --- Title-search matching helpers ---
TITLE_TOKEN_RE = re.compile(r"[a-z0-9]+")
SURNAME_TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z\-']+")


def title_tokens(text: object) -> set:
    return set(TITLE_TOKEN_RE.findall(("" if text is None else str(text)).lower()))


def title_jaccard(a: object, b: object) -> float:
    ta, tb = title_tokens(a), title_tokens(b)
    if not ta or not tb:
        return 0.0
    union = len(ta | tb)
    return (len(ta & tb) / union) if union else 0.0


def author_surnames(authors_text: object) -> set:
    raw = "" if authors_text is None else str(authors_text)
    surnames = set()
    for part in re.split(r"\s*(?:,| and | & |;)\s*", raw):
        tokens = SURNAME_TOKEN_RE.findall(part)
        if tokens:
            surnames.add(tokens[-1].lower())
    return surnames


def work_author_surnames(work: dict) -> set:
    surnames = set()
    for a in work.get("authorships", []):
        tokens = SURNAME_TOKEN_RE.findall(a.get("author", {}).get("display_name") or "")
        if tokens:
            surnames.add(tokens[-1].lower())
    return surnames


def query_openalex_batch(doi_to_cid: dict[str, str], cfg: dict) -> tuple[dict[str, dict], int]:
    """Batched DOI lookup. doi_to_cid may map several DOIs (published + arXiv) to one cid.

    Per cid, gather every returned work and keep the institution-bearing/published one.
    """
    doi_values = list(doi_to_cid.keys())
    all_cids = set(doi_to_cid.values())
    filter_value = "doi:" + "|".join(doi_values)
    params = {
        "filter": filter_value,
        "per-page": min(200, max(1, len(doi_values))),
        "mailto": cfg["OPENALEX_EMAIL"],
        "api_key": cfg["OPENALEX_API_KEY"],
    }

    retries = 0
    while True:
        try:
            resp = requests.get("https://api.openalex.org/works", params=params, timeout=OPENALEX_TIMEOUT_SECONDS)
            status_code = resp.status_code

            if status_code == 429 or status_code >= 500:
                if retries >= OPENALEX_MAX_RETRIES:
                    return ({cid: {"status": "transient_fail", "reason": f"http_{status_code}"} for cid in all_cids}, retries)
                sleep_seconds = OPENALEX_BASE_SLEEP_SECONDS * (2 ** retries)
                time.sleep(sleep_seconds)
                retries += 1
                continue

            if status_code != 200:
                return ({cid: {"status": "transient_fail", "reason": f"http_{status_code}"} for cid in all_cids}, retries)

            works = resp.json().get("results", [])

            cid_to_works: dict[str, list] = {cid: [] for cid in all_cids}
            for work in works:
                cid = doi_to_cid.get(normalize_doi(work.get("doi", "")))
                if cid is not None:
                    cid_to_works[cid].append(work)

            result_map = {}
            for cid in all_cids:
                cand_works = cid_to_works[cid]
                if not cand_works:
                    result_map[cid] = {"status": "permanent_fail", "reason": "not_found"}
                    continue

                best, best_inst = pick_best_work(cand_works)
                if best is not None and best_inst:
                    is_arxiv = normalize_doi(best.get("doi", "")).startswith("10.48550/arxiv.")
                    result_map[cid] = {
                        "status": "ok",
                        "reason": "ok",
                        "institution": best_inst["institution"],
                        "country_code": best_inst["country_code"],
                        "institution_source": "openalex_arxiv_doi" if is_arxiv else "openalex_published_doi",
                        "confidence_tier": "A",
                    }
                else:
                    result_map[cid] = {"status": "permanent_fail", "reason": "no_institution"}

            return (result_map, retries)

        except requests.RequestException as e:
            if retries >= OPENALEX_MAX_RETRIES:
                return ({cid: {"status": "transient_fail", "reason": f"exception_{type(e).__name__}"} for cid in all_cids}, retries)
            sleep_seconds = OPENALEX_BASE_SLEEP_SECONDS * (2 ** retries)
            time.sleep(sleep_seconds)
            retries += 1


def query_openalex_by_title(title: object, year: object, surnames: set, cfg: dict) -> tuple[dict, int]:
    """Fallback: find the published version of a paper by title search and validate the match."""
    title_text = ("" if title is None else str(title)).strip()
    clean_title = re.sub(r"\s+", " ", re.sub(r"[^A-Za-z0-9 ]+", " ", title_text)).strip()
    if not clean_title:
        return ({"status": "permanent_fail", "reason": "no_institution"}, 0)

    filter_value = "title.search:" + clean_title
    try:
        y = int(year)
        filter_value += f",publication_year:{y - 1}-{y + 1}"
    except (TypeError, ValueError):
        pass

    params = {
        "filter": filter_value,
        "per-page": 10,
        "mailto": cfg["OPENALEX_EMAIL"],
        "api_key": cfg["OPENALEX_API_KEY"],
    }

    retries = 0
    while True:
        try:
            resp = requests.get("https://api.openalex.org/works", params=params, timeout=OPENALEX_TIMEOUT_SECONDS)
            status_code = resp.status_code

            if status_code == 429 or status_code >= 500:
                if retries >= OPENALEX_MAX_RETRIES:
                    return ({"status": "transient_fail", "reason": f"http_{status_code}"}, retries)
                time.sleep(OPENALEX_BASE_SLEEP_SECONDS * (2 ** retries))
                retries += 1
                continue

            if status_code != 200:
                return ({"status": "transient_fail", "reason": f"http_{status_code}"}, retries)

            matched_works = []
            for work in resp.json().get("results", []):
                if title_jaccard(title_text, work.get("title", "")) < OPENALEX_TITLE_MATCH_MIN_JACCARD:
                    continue
                if surnames and not (surnames & work_author_surnames(work)):
                    continue
                matched_works.append(work)

            best, best_inst = pick_best_work(matched_works)
            if best is not None and best_inst:
                return ({
                    "status": "ok",
                    "reason": "ok",
                    "institution": best_inst["institution"],
                    "country_code": best_inst["country_code"],
                    "institution_source": "openalex_title_search",
                    "confidence_tier": "B",
                }, retries)

            return ({"status": "permanent_fail", "reason": "no_institution"}, retries)

        except requests.RequestException as e:
            if retries >= OPENALEX_MAX_RETRIES:
                return ({"status": "transient_fail", "reason": f"exception_{type(e).__name__}"}, retries)
            time.sleep(OPENALEX_BASE_SLEEP_SECONDS * (2 ** retries))
            retries += 1


def _read_json_dict(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_gate_cache_safe(path: str) -> dict:
    if not os.path.exists(path):
        return {}

    try:
        return _read_json_dict(path)
    except (json.JSONDecodeError, ValueError):
        broken_path = path + ".broken"
        try:
            os.replace(path, broken_path)
            print(f"[Gate cache] Corrupted cache moved to: {broken_path}")
        except OSError:
            print("[Gate cache] Corrupted cache detected; could not rename.")
        # Fall back to the previous good snapshot if available.
        bak_path = path + ".bak"
        if os.path.exists(bak_path):
            try:
                cache = _read_json_dict(bak_path)
                print(f"[Gate cache] Recovered {len(cache)} entries from backup: {bak_path}")
                return cache
            except (json.JSONDecodeError, ValueError):
                print("[Gate cache] Backup cache also unreadable; starting from empty cache.")
        return {}


def save_gate_cache(path: str, cache: dict) -> None:
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2)
        f.flush()
        os.fsync(f.fileno())  # force bytes to disk before rename (survives power loss)
    # Keep the previous good cache as a fallback before overwriting.
    if os.path.exists(path):
        try:
            os.replace(path, path + ".bak")
        except OSError:
            pass
    os.replace(tmp_path, path)


def _year_from_update_date(value: object):
    text = "" if value is None else str(value)
    return int(text[:4]) if len(text) >= 4 and text[:4].isdigit() else None


def get_meta_map_for_batch(batch_ids: list[str]) -> dict[str, dict]:
    """Fetch title/authors/published-doi/year for the given cids from the candidate pool."""
    if not batch_ids:
        return {}
    batch_id_df = pd.DataFrame({"canonical_id": batch_ids})
    duckdb.register("batch_ids_view", batch_id_df)
    meta_df = duckdb.sql(
        f"""
        SELECT
            p.canonical_id,
            COALESCE(p.title, '') AS title,
            COALESCE(CAST(p.authors AS VARCHAR), '') AS authors,
            COALESCE(CAST(p.doi AS VARCHAR), '') AS doi,
            COALESCE(CAST(p.update_date AS VARCHAR), '') AS update_date
        FROM read_parquet('{candidate_pool_path}') AS p
        INNER JOIN batch_ids_view AS b
          ON p.canonical_id = b.canonical_id
        """
    ).df()
    return {
        str(row.canonical_id): {
            "title": str(row.title),
            "authors": str(row.authors),
            "doi": str(row.doi),
            "year": _year_from_update_date(row.update_date),
        }
        for row in meta_df.itertuples(index=False)
    }


def build_doi_batches(cids: list[str], meta_map: dict[str, dict]) -> list[list[tuple[str, list[str]]]]:
    """Group cids so the total number of DOIs per request stays within the OR-filter limit."""
    batches = []
    current = []
    current_doi_count = 0
    for cid in cids:
        dois = [cid_to_arxiv_doi(cid)]
        if OPENALEX_USE_PUBLISHED_DOI:
            pub = normalize_doi(meta_map.get(cid, {}).get("doi", ""))
            if is_valid_doi(pub) and pub not in dois:
                dois.append(pub)
        if current and current_doi_count + len(dois) > OPENALEX_MAX_DOIS_PER_REQUEST:
            batches.append(current)
            current = []
            current_doi_count = 0
        current.append((cid, dois))
        current_doi_count += len(dois)
    if current:
        batches.append(current)
    return batches


def recover_selected_from_output_parquet(cache: dict) -> tuple[list[str], set[str], int]:
    selected_ids = []
    seen_selected = set()
    recovered_ok = 0

    if not os.path.exists(final_valid_corpus_path):
        return selected_ids, seen_selected, recovered_ok

    try:
        schema_df = duckdb.sql(
            f"DESCRIBE SELECT * FROM read_parquet('{final_valid_corpus_path}')"
        ).df()
        existing_cols = set(schema_df["column_name"].tolist())

        source_expr = "institution_source" if "institution_source" in existing_cols else "NULL AS institution_source"
        tier_expr = (
            "institution_confidence_tier"
            if "institution_confidence_tier" in existing_cols
            else "NULL AS institution_confidence_tier"
        )
        rank_expr = "selection_rank" if "selection_rank" in existing_cols else "0 AS selection_rank"

        existing_df = duckdb.sql(
            f"""
            SELECT
                canonical_id,
                openalex_primary_institution,
                openalex_country_code,
                {source_expr},
                {tier_expr},
                {rank_expr}
            FROM read_parquet('{final_valid_corpus_path}')
            ORDER BY selection_rank
            """
        ).df()

        now_utc = datetime.now(timezone.utc).isoformat()
        for row in existing_df.itertuples(index=False):
            cid = str(row.canonical_id)
            inst = "" if row.openalex_primary_institution is None else str(row.openalex_primary_institution).strip()
            if not inst:
                continue

            country = "UNK" if row.openalex_country_code is None else str(row.openalex_country_code).strip().upper() or "UNK"
            source = "recovered_output" if row.institution_source is None else str(row.institution_source).strip() or "recovered_output"
            tier = "unknown" if row.institution_confidence_tier is None else str(row.institution_confidence_tier).strip() or "unknown"

            cache[cid] = {
                "status": "ok",
                "reason": "recovered_from_output",
                "institution": inst,
                "country_code": country,
                "institution_source": source,
                "confidence_tier": tier,
                "updated_at_utc": now_utc,
            }

            if cid not in seen_selected:
                selected_ids.append(cid)
                seen_selected.add(cid)
                recovered_ok += 1
    except Exception as e:
        print(f"[Gate recovery] Could not recover existing selected ids: {type(e).__name__}")

    return selected_ids, seen_selected, recovered_ok


def recover_selected_from_cache(candidate_ids: list[str], cache: dict, selected_ids: list[str], seen_selected: set[str]) -> int:
    recovered_ok = 0
    for cid in candidate_ids:
        if cid in seen_selected:
            continue
        entry = cache.get(cid, {})
        if entry.get("status") == "ok" and entry.get("institution", "").strip():
            selected_ids.append(cid)
            seen_selected.add(cid)
            recovered_ok += 1
            if len(selected_ids) >= TARGET_VALID_SIZE:
                break
    return recovered_ok


def build_pending_ids(candidate_ids: list[str], cache: dict, seen_selected: set[str]) -> list[str]:
    fresh = []
    stale = []
    for cid in candidate_ids:
        if cid in seen_selected:
            continue

        entry = cache.get(cid)
        if not entry:
            fresh.append(cid)
            continue

        status = entry.get("status")
        reason = entry.get("reason")
        version = int(entry.get("schema_version", 1))

        if status == "ok":
            continue
        if status == "transient_fail":
            stale.append(cid)
            continue
        # Fails written under an older strategy deserve a fresh attempt.
        if status == "permanent_fail" and version < CACHE_SCHEMA_VERSION:
            stale.append(cid)
            continue
        if OPENALEX_RETRY_NOT_FOUND and status == "permanent_fail" and reason == "not_found":
            stale.append(cid)
            continue
        if OPENALEX_RETRY_NO_INSTITUTION and status == "permanent_fail" and reason == "no_institution":
            stale.append(cid)

    # Prioritize never-queried candidates so coverage grows before re-trying old fails.
    pending = fresh + stale
    if OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN is not None:
        lookup_cap = max(0, int(OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN))
        pending = pending[:lookup_cap]
    return pending


def materialize_output_and_report(
    selected_ids: list[str],
    cache: dict,
    counters: dict,
    candidate_count: int,
) -> tuple[pd.DataFrame, bool]:
    selected_ids = selected_ids[:TARGET_VALID_SIZE]
    selection_df = pd.DataFrame(
        {
            "canonical_id": selected_ids,
            "selection_rank": list(range(len(selected_ids))),
        }
    )
    duckdb.register("gate_selected_ids_view", selection_df)

    final_valid_df = duckdb.sql(
        f"""
        SELECT
            pool.*,
            sel.selection_rank
        FROM read_parquet('{candidate_pool_path}') AS pool
        INNER JOIN gate_selected_ids_view AS sel
          ON pool.canonical_id = sel.canonical_id
        ORDER BY sel.selection_rank
        """
    ).df()

    final_valid_df["openalex_primary_institution"] = final_valid_df["canonical_id"].map(
        lambda cid: (cache.get(str(cid), {}).get("institution") or "").strip()
    )
    final_valid_df["openalex_country_code"] = final_valid_df["canonical_id"].map(
        lambda cid: (cache.get(str(cid), {}).get("country_code") or "UNK").strip().upper() or "UNK"
    )
    final_valid_df["institution_source"] = final_valid_df["canonical_id"].map(
        lambda cid: (cache.get(str(cid), {}).get("institution_source") or "unknown").strip()
    )
    final_valid_df["institution_confidence_tier"] = final_valid_df["canonical_id"].map(
        lambda cid: (cache.get(str(cid), {}).get("confidence_tier") or "unknown").strip()
    )

    if (final_valid_df["openalex_primary_institution"].str.strip() == "").any():
        raise ValueError("Final valid corpus contains rows without institution after gate filtering")

    duckdb.register("final_valid_df_view", final_valid_df)
    duckdb.sql(f"COPY final_valid_df_view TO '{final_valid_corpus_path}' (FORMAT PARQUET);")

    gate_status_counts = pd.Series(
        [entry.get("status", "missing") for entry in cache.values()]
    ).value_counts().to_dict()
    source_counts = final_valid_df["institution_source"].value_counts().to_dict() if len(final_valid_df) else {}
    tier_counts = final_valid_df["institution_confidence_tier"].value_counts().to_dict() if len(final_valid_df) else {}

    target_reached = len(final_valid_df) >= TARGET_VALID_SIZE

    gate_report = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "target_valid_size": TARGET_VALID_SIZE,
        "target_reached": target_reached,
        "max_new_lookups_this_run": OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN,
        "candidate_pool_path": candidate_pool_path,
        "output_path": final_valid_corpus_path,
        "openalex_cache_path": openalex_gate_cache_path,
        "candidate_count": candidate_count,
        "selected_count": len(final_valid_df),
        "query_batches_this_run": counters["query_batches"],
        "query_attempts_this_run": counters["query_attempts"],
        "retried_batches_this_run": counters["retried_batches"],
        "title_searches_this_run": counters["title_searches"],
        "new_ok_this_run": counters["new_ok"],
        "new_transient_fail_this_run": counters["new_transient"],
        "new_permanent_fail_this_run": counters["new_permanent"],
        "ok_by_strategy_this_run": {
            "openalex_published_doi": counters["strategy_published_doi"],
            "openalex_arxiv_doi": counters["strategy_arxiv_doi"],
            "openalex_title_search": counters["strategy_title_search"],
        },
        "recovered_ok_from_output": counters["recovered_from_output"],
        "recovered_ok_from_cache": counters["recovered_from_cache"],
        "cache_status_counts": {k: int(v) for k, v in gate_status_counts.items()},
        "selected_source_counts": {k: int(v) for k, v in source_counts.items()},
        "selected_tier_counts": {k: int(v) for k, v in tier_counts.items()},
    }

    with open(openalex_gate_report_path, "w", encoding="utf-8") as f:
        json.dump(gate_report, f, indent=2)

    return final_valid_df, target_reached


openalex_cfg = load_openalex_config(openalex_config_path)
openalex_gate_cache = load_gate_cache_safe(openalex_gate_cache_path)

candidate_ids = duckdb.sql(
    f"""
    SELECT canonical_id
    FROM read_parquet('{candidate_pool_path}')
    ORDER BY update_date DESC, canonical_id
    """
).df()["canonical_id"].astype(str).tolist()

selected_ids, seen_selected, recovered_from_output = recover_selected_from_output_parquet(openalex_gate_cache)
recovered_from_cache = recover_selected_from_cache(candidate_ids, openalex_gate_cache, selected_ids, seen_selected)

counters = {
    "query_attempts": 0,
    "query_batches": 0,
    "retried_batches": 0,
    "new_ok": 0,
    "new_transient": 0,
    "new_permanent": 0,
    "title_searches": 0,
    "strategy_published_doi": 0,
    "strategy_arxiv_doi": 0,
    "strategy_title_search": 0,
    "recovered_from_output": recovered_from_output,
    "recovered_from_cache": recovered_from_cache,
}

print("Gate resume status:")
print(f"  recovered selected from output: {recovered_from_output}")
print(f"  recovered selected from cache: {recovered_from_cache}")
print(f"  already selected total: {len(selected_ids)}/{TARGET_VALID_SIZE}")
print(f"  cache entries: {len(openalex_gate_cache)}")

title_search_pending: list[str] = []
meta_map: dict[str, dict] = {}
last_saved_ok = 0  # tracks new_ok count at the last cache checkpoint

if len(selected_ids) < TARGET_VALID_SIZE:
    pending = build_pending_ids(candidate_ids, openalex_gate_cache, seen_selected)
    meta_map = get_meta_map_for_batch(pending)
    doi_batches = build_doi_batches(pending, meta_map)

    # --- Phase A: batched multi-DOI lookup (published DOI + arXiv DOI) ---
    if tqdm is not None:
        iterator = tqdm(doi_batches, desc="OpenAlex DOI gate", unit="batch")
    else:
        iterator = doi_batches

    for batch in iterator:
        if len(selected_ids) >= TARGET_VALID_SIZE:
            break

        doi_to_cid = {doi: cid for cid, dois in batch for doi in dois}

        batch_result, retries_used = query_openalex_batch(doi_to_cid, openalex_cfg)
        counters["query_attempts"] += len(batch)
        counters["query_batches"] += 1
        counters["retried_batches"] += int(retries_used > 0)

        now_utc = datetime.now(timezone.utc).isoformat()
        for cid, info in batch_result.items():
            openalex_gate_cache[cid] = {
                **info,
                "schema_version": CACHE_SCHEMA_VERSION,
                "updated_at_utc": now_utc,
            }

            if info.get("status") == "ok" and info.get("institution", "").strip():
                counters["new_ok"] += 1
                source = info.get("institution_source", "")
                if source == "openalex_published_doi":
                    counters["strategy_published_doi"] += 1
                elif source == "openalex_arxiv_doi":
                    counters["strategy_arxiv_doi"] += 1
                if cid not in seen_selected:
                    selected_ids.append(cid)
                    seen_selected.add(cid)
                    if len(selected_ids) >= TARGET_VALID_SIZE:
                        break
            elif info.get("status") == "transient_fail":
                counters["new_transient"] += 1
            else:
                counters["new_permanent"] += 1
                if OPENALEX_ENABLE_TITLE_SEARCH_FALLBACK:
                    title_search_pending.append(cid)

        if tqdm is not None and hasattr(iterator, "set_postfix"):
            iterator.set_postfix(
                selected=f"{len(selected_ids)}/{TARGET_VALID_SIZE}",
                attempts=counters["query_attempts"],
            )
        elif counters["query_batches"] % 20 == 0 or len(selected_ids) >= TARGET_VALID_SIZE:
            print(
                f"DOI gate progress: batches={counters['query_batches']}, attempts={counters['query_attempts']}, selected={len(selected_ids)}/{TARGET_VALID_SIZE}"
            )

        if (
            counters["new_ok"] - last_saved_ok >= OPENALEX_CACHE_CHECKPOINT_EVERY_OK
            or counters["query_batches"] % OPENALEX_CACHE_CHECKPOINT_EVERY_BATCHES == 0
        ):
            save_gate_cache(openalex_gate_cache_path, openalex_gate_cache)
            last_saved_ok = counters["new_ok"]

        time.sleep(OPENALEX_INTER_BATCH_SLEEP_SECONDS)

    # --- Phase B: bounded title-search fallback for DOI misses ---
    if (
        OPENALEX_ENABLE_TITLE_SEARCH_FALLBACK
        and len(selected_ids) < TARGET_VALID_SIZE
        and title_search_pending
    ):
        budget = max(0, int(OPENALEX_MAX_TITLE_SEARCHES_THIS_RUN))
        ts_ids = title_search_pending[:budget]
        ts_iterator = (
            tqdm(ts_ids, desc="OpenAlex title gate", unit="paper") if tqdm is not None else ts_ids
        )

        for cid in ts_iterator:
            if len(selected_ids) >= TARGET_VALID_SIZE:
                break

            meta = meta_map.get(cid, {})
            surnames = author_surnames(meta.get("authors", ""))
            info, retries_used = query_openalex_by_title(
                meta.get("title", ""), meta.get("year"), surnames, openalex_cfg
            )
            counters["title_searches"] += 1
            counters["retried_batches"] += int(retries_used > 0)

            now_utc = datetime.now(timezone.utc).isoformat()
            openalex_gate_cache[cid] = {
                **info,
                "schema_version": CACHE_SCHEMA_VERSION,
                "updated_at_utc": now_utc,
            }

            if info.get("status") == "ok" and info.get("institution", "").strip():
                counters["new_ok"] += 1
                counters["strategy_title_search"] += 1
                if cid not in seen_selected:
                    selected_ids.append(cid)
                    seen_selected.add(cid)

            if tqdm is not None and hasattr(ts_iterator, "set_postfix"):
                ts_iterator.set_postfix(
                    selected=f"{len(selected_ids)}/{TARGET_VALID_SIZE}",
                    title_searches=counters["title_searches"],
                )

            if (
                counters["new_ok"] - last_saved_ok >= OPENALEX_CACHE_CHECKPOINT_EVERY_OK
                or counters["title_searches"] % OPENALEX_CACHE_CHECKPOINT_EVERY_BATCHES == 0
            ):
                save_gate_cache(openalex_gate_cache_path, openalex_gate_cache)
                last_saved_ok = counters["new_ok"]

            time.sleep(OPENALEX_INTER_BATCH_SLEEP_SECONDS)

save_gate_cache(openalex_gate_cache_path, openalex_gate_cache)

final_valid_df, target_reached = materialize_output_and_report(
    selected_ids=selected_ids,
    cache=openalex_gate_cache,
    counters=counters,
    candidate_count=candidate_count,
)

final_count = len(final_valid_df)
print("cs_candidate_pool written:", candidate_pool_path)
print("final_50k_with_institution written:", final_valid_corpus_path)
print("openalex_gate_cache written:", openalex_gate_cache_path)
print("openalex_gate_report written:", openalex_gate_report_path)
print("Candidate pool size:", candidate_count)
print("Final valid corpus size this run:", final_count)
if not target_reached:
    print(
        "Target not reached yet. Increase OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN or rerun this cell to continue from cache."
    )

Gate run config:
  OPENALEX_MAX_NEW_LOOKUPS_THIS_RUN: None
  OPENALEX_CACHE_CHECKPOINT_EVERY_OK: 5000
  OPENALEX_CACHE_CHECKPOINT_EVERY_BATCHES: 100
  OPENALEX_RETRY_NOT_FOUND: False
  OPENALEX_RETRY_NO_INSTITUTION: False
  OPENALEX_USE_PUBLISHED_DOI: True
  OPENALEX_ENABLE_TITLE_SEARCH_FALLBACK: False
  OPENALEX_MAX_TITLE_SEARCHES_THIS_RUN: 2000
  CACHE_SCHEMA_VERSION: 2
Gate resume status:
  recovered selected from output: 2218
  recovered selected from cache: 8608
  already selected total: 10826/50000
  cache entries: 28567


OpenAlex DOI gate:  34%|███▍      | 7398/21687 [2:27:07<4:44:09,  1.19s/batch, attempts=330230, selected=50000/50000] 


cs_candidate_pool written: data\cs_candidate_pool.parquet
final_50k_with_institution written: data\final_50k_with_institution.parquet
openalex_gate_cache written: data\openalex_gate_cache.json
openalex_gate_report written: data\openalex_gate_report.json
Candidate pool size: 955379
Final valid corpus size this run: 50000


In [5]:
# --------------------------
# Write selection report + preview
# --------------------------
selection_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_source": dataset_source,
    "dataset_path": dataset_path,
    "json_file_path": json_file_path,
    "save_eligible_pool": SAVE_ELIGIBLE_POOL,
    "eligible_pool_path": eligible_pool_path if SAVE_ELIGIBLE_POOL else None,
    "candidate_pool_path": candidate_pool_path,
    "final_corpus_path": final_valid_corpus_path,
    "openalex_gate_cache_path": openalex_gate_cache_path,
    "openalex_gate_report_path": openalex_gate_report_path,
    "target_valid_size": TARGET_VALID_SIZE,
    "sample_seed": SAMPLE_SEED,
    "counts": {
        "total_records": int(selection_stats["total_records"]),
        "cs_records": int(selection_stats["cs_records"]),
        "cs_ids_with_version_suffix": int(selection_stats["cs_ids_with_version_suffix"]),
        "dropped_missing_fields": int(selection_stats["dropped_missing_fields"]),
        "cs_with_non_empty_text": int(selection_stats["cs_with_non_empty_text"]),
        "dropped_duplicates": int(selection_stats["dropped_duplicates"]),
        "candidate_pool_size": candidate_count,
        "final_valid_corpus_size": final_count,
    },
    "rules": {
        "cs_filter": "categories LIKE 'cs.%' OR categories LIKE '% cs.%'",
        "text_filter": "title and abstract non-null and non-empty",
        "dedup": "canonical_id = regexp_replace(id, 'v[0-9]+$', ''), keep latest update_date",
        "openalex_gate": "batch DOI lookup with transient retry and institution-required selection",
    },
}

with open(selection_report_path, "w", encoding="utf-8") as f:
    json.dump(selection_report, f, indent=2)

preview_df = duckdb.sql(
    f"""
    SELECT id, canonical_id, openalex_primary_institution, openalex_country_code
    FROM read_parquet('{final_valid_corpus_path}')
    LIMIT 5;
    """
).df()

print("corpus_selection_report written:", selection_report_path)
print("\nPhase 1 complete.")
print("Gated sample preview:")
print(preview_df)

corpus_selection_report written: data\corpus_selection_report.json

Phase 1 complete.
Gated sample preview:
           id canonical_id                       openalex_primary_institution  \
0  2502.19947   2502.19947  Laboratoire d’Analyse et de Mathématiques Appl...   
1  2502.20914   2502.20914             Laboratoire d'Informatique de Grenoble   
2  2503.11910   2503.11910       Skolkovo Institute of Science and Technology   
3  2507.00460   2507.00460                           Wichita State University   
4  2605.13587   2605.13587  Centre de Coopération Internationale en Recher...   

  openalex_country_code  
0                    FR  
1                    FR  
2                    RU  
3                    US  
4                    FR  


## Phase 2. Text Cleaning and Metadata Normalization

- **Goal:** Convert the 50K corpus into clean, reproducible documents ready for embedding.
- **Tasks:** Clean title/abstract text, retain both raw and cleaned fields, normalize category/author fields, extract year, and generate stable document IDs.
- **Expected outputs:** `final_50k_cleaned.parquet`, `metadata_schema.md`, `text_cleaning_report.json`.

In [6]:
# --------------------------
# Phase 2 config + IO paths
# --------------------------
import hashlib
import re

import pandas as pd

phase2_input_path = final_valid_corpus_path
phase2_output_path = os.path.join(DATA_DIR, "final_50k_cleaned.parquet")
phase2_report_path = os.path.join(DATA_DIR, "text_cleaning_report.json")

EXPECTED_INPUT_ROWS = TARGET_VALID_SIZE

if not os.path.exists(phase2_input_path):
    raise FileNotFoundError(f"Phase 2 input not found: {phase2_input_path}")

phase2_df = duckdb.sql(f"SELECT * FROM read_parquet('{phase2_input_path}')").df()

if len(phase2_df) != EXPECTED_INPUT_ROWS:
    raise ValueError(
        f"Unexpected input rows: expected {EXPECTED_INPUT_ROWS}, got {len(phase2_df)}"
    )

print("Phase 2 input:", phase2_input_path)
print("Phase 2 output:", phase2_output_path)
print("Phase 2 report:", phase2_report_path)
print("Input rows:", len(phase2_df))

Phase 2 input: data\final_50k_with_institution.parquet
Phase 2 output: data\final_50k_cleaned.parquet
Phase 2 report: data\text_cleaning_report.json
Input rows: 50000


In [7]:
# --------------------------
# Text cleaning (conservative)
# --------------------------
HTML_TAG_RE = re.compile(r"<[^>]+>")
CONTROL_CHAR_RE = re.compile(r"[\x00-\x1f\x7f]")
WHITESPACE_RE = re.compile(r"\s+")


def clean_text(value: object) -> str:
    if value is None:
        return ""
    text = str(value)
    text = HTML_TAG_RE.sub(" ", text)
    text = CONTROL_CHAR_RE.sub(" ", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


phase2_df["cleaned_title"] = phase2_df["title"].map(clean_text)
phase2_df["cleaned_abstract"] = phase2_df["abstract"].map(clean_text)

print("Text cleaning finished.")
print("Empty cleaned_title:", int((phase2_df["cleaned_title"] == "").sum()))
print("Empty cleaned_abstract:", int((phase2_df["cleaned_abstract"] == "").sum()))

Text cleaning finished.
Empty cleaned_title: 0
Empty cleaned_abstract: 0


In [8]:
# --------------------------
# Metadata normalization
# --------------------------

def normalize_authors(value: object) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        joined = ", ".join(str(x).strip() for x in value if str(x).strip())
    else:
        joined = str(value)
    joined = WHITESPACE_RE.sub(" ", joined).strip()
    return joined


def parse_category_tokens(value: object) -> list[str]:
    if value is None:
        return []
    tokens = [token.strip() for token in str(value).split(" ") if token.strip()]
    return tokens


def pick_primary_category(tokens: list[str]) -> str:
    for token in tokens:
        if token.startswith("cs."):
            return token
    return tokens[0] if tokens else "unknown"


phase2_df["normalized_authors"] = phase2_df["authors"].map(normalize_authors)
phase2_df["category_tokens"] = phase2_df["categories"].map(parse_category_tokens)
phase2_df["primary_category"] = phase2_df["category_tokens"].map(pick_primary_category)
phase2_df["year"] = pd.to_datetime(phase2_df["update_date"], errors="coerce").dt.year.astype("Int64")

print("Metadata normalization finished.")
print("Missing year:", int(phase2_df["year"].isna().sum()))
print("Unknown primary_category:", int((phase2_df["primary_category"] == "unknown").sum()))

Metadata normalization finished.
Missing year: 0
Unknown primary_category: 0


In [9]:
# --------------------------
# Stable document_id + document_text
# --------------------------

def canonical_arxiv_id(value: object) -> str:
    if value is None:
        return ""
    return re.sub(r"v[0-9]+$", "", str(value).strip())


def build_document_id(row: pd.Series) -> str:
    canonical_id = row["canonical_id"]
    if canonical_id:
        return canonical_id
    fallback_source = f"{row['cleaned_title']}\n\n{row['cleaned_abstract']}"
    fallback_hash = hashlib.sha256(fallback_source.encode("utf-8")).hexdigest()[:16]
    return f"fallback-{fallback_hash}"


phase2_df["canonical_id"] = phase2_df["id"].map(canonical_arxiv_id)
phase2_df["document_text"] = phase2_df["cleaned_title"] + "\n\n" + phase2_df["cleaned_abstract"]
phase2_df["document_id"] = phase2_df.apply(build_document_id, axis=1)

# collision-safe suffix only if duplicates exist
if phase2_df["document_id"].duplicated().any():
    phase2_df["document_id"] = (
        phase2_df["document_id"]
        + "-"
        + phase2_df.groupby("document_id").cumcount().astype(str)
    )

if phase2_df["document_id"].duplicated().any():
    raise ValueError("document_id is not unique after collision handling")

if (phase2_df["document_text"].str.strip() == "").any():
    raise ValueError("document_text contains empty records")

print("document_id/document_text build finished.")
print("Unique document_id count:", phase2_df["document_id"].nunique())

document_id/document_text build finished.
Unique document_id count: 50000


In [10]:
# --------------------------
# Quality validation
# --------------------------
row_count = len(phase2_df)
if row_count != EXPECTED_INPUT_ROWS:
    raise ValueError(f"Row count drift: expected {EXPECTED_INPUT_ROWS}, got {row_count}")

null_counts = {
    "cleaned_title": int(phase2_df["cleaned_title"].isna().sum() + (phase2_df["cleaned_title"].str.strip() == "").sum()),
    "cleaned_abstract": int(phase2_df["cleaned_abstract"].isna().sum() + (phase2_df["cleaned_abstract"].str.strip() == "").sum()),
    "year": int(phase2_df["year"].isna().sum()),
    "primary_category": int(phase2_df["primary_category"].isna().sum() + (phase2_df["primary_category"].str.strip() == "").sum()),
    "document_id": int(phase2_df["document_id"].isna().sum() + (phase2_df["document_id"].str.strip() == "").sum()),
}

length_stats = {
    "cleaned_title_chars": {
        "min": int(phase2_df["cleaned_title"].str.len().min()),
        "median": float(phase2_df["cleaned_title"].str.len().median()),
        "max": int(phase2_df["cleaned_title"].str.len().max()),
    },
    "cleaned_abstract_chars": {
        "min": int(phase2_df["cleaned_abstract"].str.len().min()),
        "median": float(phase2_df["cleaned_abstract"].str.len().median()),
        "max": int(phase2_df["cleaned_abstract"].str.len().max()),
    },
}

year_min = int(phase2_df["year"].min()) if phase2_df["year"].notna().any() else None
year_max = int(phase2_df["year"].max()) if phase2_df["year"].notna().any() else None

top_primary_categories = (
    phase2_df["primary_category"].value_counts().head(10).to_dict()
)

phase2_validation = {
    "row_count": row_count,
    "expected_rows": EXPECTED_INPUT_ROWS,
    "document_id_unique": bool(phase2_df["document_id"].nunique() == row_count),
    "null_or_empty_counts": null_counts,
    "length_stats": length_stats,
    "year_range": {"min": year_min, "max": year_max},
    "top_primary_categories": {k: int(v) for k, v in top_primary_categories.items()},
}

print("Validation passed for row count and document_id uniqueness.")
print("Null/empty key fields:", null_counts)
print("Year range:", phase2_validation["year_range"])
print("Top categories (sample):", list(phase2_validation["top_primary_categories"].items())[:5])

phase2_df[["document_id", "title", "cleaned_title", "primary_category", "year"]].head(5)

Validation passed for row count and document_id uniqueness.
Null/empty key fields: {'cleaned_title': 0, 'cleaned_abstract': 0, 'year': 0, 'primary_category': 0, 'document_id': 0}
Year range: {'min': 2024, 'max': 2026}
Top categories (sample): [('cs.NA', 8226), ('cs.LG', 6952), ('cs.CV', 5662), ('cs.SY', 3868), ('cs.RO', 2822)]


,document_id,title,cleaned_title,primary_category,year
0,2502.19947,Numerical analysis of a finite volume method f...,Numerical analysis of a finite volume method f...,cs.NA,2026
1,2502.20914,"Everything, Everywhere, All at Once: Is Mechan...","Everything, Everywhere, All at Once: Is Mechan...",cs.LG,2026
2,2503.11910,RTD-Lite: Scalable Topological Analysis for Co...,RTD-Lite: Scalable Topological Analysis for Co...,cs.LG,2026
3,2507.00460,Pitfalls of Evaluating Language Models with Op...,Pitfalls of Evaluating Language Models with Op...,cs.CL,2026
4,2605.13587,Reframing preprocessing selection as model-int...,Reframing preprocessing selection as model-int...,cs.LG,2026


In [11]:
# --------------------------
# Export cleaned dataset + report
# --------------------------
phase2_output_columns = [
    "document_id",
    "document_text",
    "canonical_id",
    "id",
    "title",
    "abstract",
    "cleaned_title",
    "cleaned_abstract",
    "authors",
    "normalized_authors",
    "categories",
    "category_tokens",
    "primary_category",
    "update_date",
    "year",
]

optional_columns = [
    "openalex_primary_institution",
    "openalex_country_code",
    "institution_source",
    "institution_confidence_tier",
    "selection_rank",
]
phase2_output_columns.extend([c for c in optional_columns if c in phase2_df.columns])

phase2_output_df = phase2_df[phase2_output_columns].copy()

duckdb.register("phase2_output_df_view", phase2_output_df)
duckdb.sql(
    f"""
    COPY phase2_output_df_view
    TO '{phase2_output_path}' (FORMAT PARQUET);
    """
)

phase2_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_path": phase2_input_path,
    "output_path": phase2_output_path,
    "row_count": len(phase2_output_df),
    "optional_columns_kept": [c for c in optional_columns if c in phase2_output_df.columns],
    "cleaning_rules": {
        "html_tags_removed": True,
        "control_characters_removed": True,
        "whitespace_normalized": True,
    },
    "normalization_rules": {
        "year_from": "update_date",
        "primary_category": "first cs.* token, fallback first token, else unknown",
        "document_id": "canonical arxiv id (version suffix removed), hash fallback for empty ids",
    },
    "validation": phase2_validation,
}

with open(phase2_report_path, "w", encoding="utf-8") as f:
    json.dump(phase2_report, f, indent=2)

print("Phase 2 output written:", phase2_output_path)
print("Phase 2 report written:", phase2_report_path)
print("Output rows:", len(phase2_output_df))

Phase 2 output written: data\final_50k_cleaned.parquet
Phase 2 report written: data\text_cleaning_report.json
Output rows: 50000


## Phase 3. Institution and Geography Enrichment with Fairness Labels

- **Goal:** Attach institution/location metadata and fairness labels on top of the OpenAlex institution already resolved in Phase 1.
- **Tasks:** Carry the Phase 1 OpenAlex institution/country through, canonicalize it against the QS Top-50 CS list (handling OpenAlex name variants), map country to region, and assign labels: privileged (QS Top-50 CS universities), underrepresented (any other resolved institution), unknown (no resolvable affiliation; expected to be ~0 because Phase 1 gates on institution presence).
- **Note:** Institution lookup now lives in Phase 1's OpenAlex gate, so Phase 3 does no API calls and no author-text fallback.
- **Expected outputs:** `final_50k_labeled.parquet`, `institution_alias_map.csv`, `label_distribution.csv`, `affiliation_match_report.json`.

In [14]:
# --------------------------
# Phase 3 config + IO validation (hybrid-first)
# --------------------------
phase3_input_path = os.path.join(DATA_DIR, "final_50k_cleaned.parquet")
phase3_output_path = os.path.join(DATA_DIR, "final_50k_labeled.parquet")
phase3_alias_map_path = os.path.join(DATA_DIR, "institution_alias_map.csv")
phase3_label_dist_path = os.path.join(DATA_DIR, "label_distribution.csv")
phase3_report_path = os.path.join(DATA_DIR, "affiliation_match_report.json")
phase3_qs_path = os.path.abspath(os.path.join(os.getcwd(), "..", "qs-top50-cs-universities.txt"))

PHASE3_EXPECTED_ROWS = TARGET_VALID_SIZE
PHASE3_REQUIRED_COLUMNS = [
    "document_id",
    "primary_category",
    "year",
    "openalex_primary_institution",
    "openalex_country_code",
]

if not os.path.exists(phase3_input_path):
    raise FileNotFoundError(f"Phase 3 input not found: {phase3_input_path}")
if not os.path.exists(phase3_qs_path):
    raise FileNotFoundError(f"QS Top50 file not found: {phase3_qs_path}")

phase3_df = duckdb.sql(f"SELECT * FROM read_parquet('{phase3_input_path}')").df()

if len(phase3_df) != PHASE3_EXPECTED_ROWS:
    raise ValueError(f"Unexpected row count: expected {PHASE3_EXPECTED_ROWS}, got {len(phase3_df)}")

missing_cols = [c for c in PHASE3_REQUIRED_COLUMNS if c not in phase3_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for Phase 3: {missing_cols}")

print("Phase 3 input:", phase3_input_path)
print("Phase 3 qs list:", phase3_qs_path)
print("Rows:", len(phase3_df))

Phase 3 input: data\final_50k_cleaned.parquet
Phase 3 qs list: c:\Users\Darjeel1ng\OneDrive - Northeastern University\Attachments\cs 6200\project\qs-top50-cs-universities.txt
Rows: 50000


In [15]:
# --------------------------
# Load QS Top50 list + build an OpenAlex-name -> QS canonical matcher
# --------------------------


def normalize_inst_key(text: object) -> str:
    """Normalize an institution name to a comparison key.

    Handles the cross-source differences seen between QS names and OpenAlex
    display names: '&' vs 'and', parenthetical aliases, punctuation, and a
    leading article ('The University of X' vs 'University of X').
    """
    s = "" if text is None else str(text)
    s = s.lower().replace("&", " and ")
    s = re.sub(r"\([^\)]*\)", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s.startswith("the "):
        s = s[4:]
    return s


qs_rows = []
with open(phase3_qs_path, "r", encoding="utf-8") as f:
    for line in f:
        raw = line.strip()
        if not raw:
            continue
        m = re.match(r"^(.*?)\s*\((.*?)\)\s*$", raw)
        if m:
            full_name = m.group(1).strip()
            alias = m.group(2).strip()
        else:
            full_name = raw
            alias = ""
        qs_rows.append({"full_name": full_name, "alias": alias, "raw": raw})

qs_top50_df = pd.DataFrame(qs_rows)
if len(qs_top50_df) != 50:
    raise ValueError(f"QS list size mismatch: expected 50, got {len(qs_top50_df)}")

# OpenAlex display names that do NOT reduce to the QS full name even after the
# normalization above (verified against the Phase 1 gate output / qs_match_debug).
# Maps an OpenAlex-side key -> QS canonical full name.
QS_OPENALEX_VARIANTS = {
    "eth zurich": "ETH Zurich \u2013 Swiss Federal Institute of Technology",
    "nanyang technological university": "Nanyang Technological University, Singapore",
    "unsw sydney": "The University of New South Wales Sydney",
}

# Exact-key matcher: a paper is Top50 only if its normalized OpenAlex name equals
# a QS full-name key or a curated variant key. No loose substring matching, so
# "University of Science and Technology of China" never collides with a QS entry.
qs_canonical_by_key: dict[str, str] = {}
for row in qs_top50_df.itertuples(index=False):
    full_key = normalize_inst_key(row.full_name)
    if full_key:
        qs_canonical_by_key[full_key] = row.full_name

for variant_key, canonical_full in QS_OPENALEX_VARIANTS.items():
    nk = normalize_inst_key(variant_key)
    if nk:
        qs_canonical_by_key.setdefault(nk, canonical_full)

print("Loaded QS entries:", len(qs_top50_df))
print("QS match keys (full names + variants):", len(qs_canonical_by_key))
print(qs_top50_df.head(5))

Loaded QS entries: 50
QS match keys (full names + variants): 53
                               full_name     alias  \
0  Massachusetts Institute of Technology       MIT   
1                    Stanford University  Stanford   
2             Carnegie Mellon University       CMU   
3       National University of Singapore       NUS   
4                   University of Oxford    Oxford   

                                           raw  
0  Massachusetts Institute of Technology (MIT)  
1               Stanford University (Stanford)  
2             Carnegie Mellon University (CMU)  
3       National University of Singapore (NUS)  
4                University of Oxford (Oxford)  


In [16]:
# --------------------------
# Institution from Phase 1 OpenAlex gate + QS Top50 canonicalization
# --------------------------
# Phase 1 already resolved an institution for every selected paper, so this stage
# is a pure pass-through plus deterministic Top50 matching (no API, no author text).
gate_inst_series = phase3_df["openalex_primary_institution"].fillna("").astype(str).str.strip()
if gate_inst_series.eq("").any():
    raise ValueError(
        "Found rows without OpenAlex institution. Phase 1 gate is expected to guarantee 100% coverage."
    )

phase3_df["primary_institution_raw"] = gate_inst_series
phase3_df["primary_institution_normalized"] = gate_inst_series
phase3_df["institution_match_key"] = phase3_df["canonical_id"].astype(str)


def match_qs_top50(inst_name: object) -> str:
    return qs_canonical_by_key.get(normalize_inst_key(inst_name), "")


phase3_df["top50_canonical_institution"] = gate_inst_series.map(match_qs_top50)
phase3_df["is_top50_institution"] = phase3_df["top50_canonical_institution"].ne("")
phase3_df["institution_match_method"] = phase3_df["is_top50_institution"].map(
    lambda hit: "qs_top50_match" if hit else "openalex_gate"
)

matched_rows = phase3_df[phase3_df["is_top50_institution"]]
if len(matched_rows) > 0:
    alias_map_df = (
        matched_rows.groupby(["primary_institution_normalized", "top50_canonical_institution"])
        .size()
        .reset_index(name="matched_paper_count")
        .sort_values("matched_paper_count", ascending=False)
    )
else:
    alias_map_df = pd.DataFrame(
        columns=["primary_institution_normalized", "top50_canonical_institution", "matched_paper_count"]
    )

print("Institution pass-through from OpenAlex complete.")
print("Rows with institution (OpenAlex gate):", int(gate_inst_series.ne("").sum()))
print("Rows matched to QS Top50:", int(phase3_df["is_top50_institution"].sum()))
print("Distinct OpenAlex institutions:", int(phase3_df["primary_institution_normalized"].nunique()))
print(alias_map_df.head(10))

Institution pass-through from OpenAlex complete.
Rows with institution (OpenAlex gate): 50000
Rows matched to QS Top50: 8553
Distinct OpenAlex institutions: 5937
             primary_institution_normalized  \
25           Technical University of Munich   
29                      Tsinghua University   
6                                ETH Zurich   
1                Carnegie Mellon University   
41  University of Illinois Urbana-Champaign   
5            Delft University of Technology   
12        KTH Royal Institute of Technology   
7           Georgia Institute of Technology   
15    Massachusetts Institute of Technology   
19                        Peking University   

                          top50_canonical_institution  matched_paper_count  
25                     Technical University of Munich                  353  
29                                Tsinghua University                  316  
6   ETH Zurich – Swiss Federal Institute of Techno...                  312  
1           

In [17]:
# --------------------------
# Country/region mapping + fairness labels
# --------------------------
TOP50_COUNTRY_REGION = {
    "Massachusetts Institute of Technology": ("US", "North America"),
    "Stanford University": ("US", "North America"),
    "Carnegie Mellon University": ("US", "North America"),
    "National University of Singapore": ("SG", "Asia"),
    "University of Oxford": ("GB", "Europe"),
    "University of California, Berkeley": ("US", "North America"),
    "Harvard University": ("US", "North America"),
    "University of Cambridge": ("GB", "Europe"),
    "ETH Zurich – Swiss Federal Institute of Technology": ("CH", "Europe"),
    "École Polytechnique Fédérale de Lausanne": ("CH", "Europe"),
    "Tsinghua University": ("CN", "Asia"),
    "Imperial College London": ("GB", "Europe"),
    "Nanyang Technological University, Singapore": ("SG", "Asia"),
    "University of Toronto": ("CA", "North America"),
    "Princeton University": ("US", "North America"),
    "Peking University": ("CN", "Asia"),
    "The Hong Kong University of Science and Technology": ("HK", "Asia"),
    "University of Washington": ("US", "North America"),
    "University College London": ("GB", "Europe"),
    "University of Illinois Urbana-Champaign": ("US", "North America"),
    "Cornell University": ("US", "North America"),
    "University of California, Los Angeles": ("US", "North America"),
    "Columbia University": ("US", "North America"),
    "The University of Edinburgh": ("GB", "Europe"),
    "University of Waterloo": ("CA", "North America"),
    "Technical University of Munich": ("DE", "Europe"),
    "The University of Tokyo": ("JP", "Asia"),
    "Georgia Institute of Technology": ("US", "North America"),
    "The University of Hong Kong": ("HK", "Asia"),
    "University of Michigan-Ann Arbor": ("US", "North America"),
    "California Institute of Technology": ("US", "North America"),
    "New York University": ("US", "North America"),
    "Seoul National University": ("KR", "Asia"),
    "The University of Texas at Austin": ("US", "North America"),
    "University of California, San Diego": ("US", "North America"),
    "The Hong Kong Polytechnic University": ("HK", "Asia"),
    "University of British Columbia": ("CA", "North America"),
    "Korea Advanced Institute of Science & Technology": ("KR", "Asia"),
    "The University of Melbourne": ("AU", "Oceania"),
    "KTH Royal Institute of Technology": ("SE", "Europe"),
    "The University of New South Wales Sydney": ("AU", "Oceania"),
    "The Chinese University of Hong Kong": ("HK", "Asia"),
    "University of Amsterdam": ("NL", "Europe"),
    "Yale University": ("US", "North America"),
    "University of Pennsylvania": ("US", "North America"),
    "Shanghai Jiao Tong University": ("CN", "Asia"),
    "Karlsruhe Institute of Technology": ("DE", "Europe"),
    "Politecnico di Milano": ("IT", "Europe"),
    "University of Maryland, College Park": ("US", "North America"),
    "Delft University of Technology": ("NL", "Europe"),
}

# The corpus is global now (institution/country come straight from OpenAlex), so
# the country -> region map must cover far more than the QS Top50 countries.
COUNTRY_TO_REGION = {
    # North America
    "US": "North America", "CA": "North America", "MX": "North America",
    "GT": "North America", "CR": "North America", "PA": "North America", "CU": "North America",
    "DO": "North America", "JM": "North America", "HN": "North America", "NI": "North America",
    "SV": "North America", "BZ": "North America", "BS": "North America", "TT": "North America",
    "HT": "North America", "PR": "North America",
    # South America
    "BR": "South America", "AR": "South America", "CL": "South America", "CO": "South America",
    "PE": "South America", "VE": "South America", "EC": "South America", "UY": "South America",
    "PY": "South America", "BO": "South America", "GY": "South America", "SR": "South America",
    # Europe
    "GB": "Europe", "IE": "Europe", "FR": "Europe", "DE": "Europe", "NL": "Europe", "BE": "Europe",
    "LU": "Europe", "CH": "Europe", "AT": "Europe", "IT": "Europe", "ES": "Europe", "PT": "Europe",
    "SE": "Europe", "NO": "Europe", "DK": "Europe", "FI": "Europe", "IS": "Europe", "PL": "Europe",
    "CZ": "Europe", "SK": "Europe", "HU": "Europe", "RO": "Europe", "BG": "Europe", "GR": "Europe",
    "HR": "Europe", "SI": "Europe", "RS": "Europe", "BA": "Europe", "MK": "Europe", "AL": "Europe",
    "ME": "Europe", "EE": "Europe", "LV": "Europe", "LT": "Europe", "UA": "Europe", "BY": "Europe",
    "MD": "Europe", "RU": "Europe", "MT": "Europe", "CY": "Europe", "LI": "Europe", "MC": "Europe",
    "AD": "Europe", "SM": "Europe", "VA": "Europe", "XK": "Europe",
    # Asia (incl. Middle East and Central Asia)
    "CN": "Asia", "JP": "Asia", "KR": "Asia", "IN": "Asia", "SG": "Asia", "HK": "Asia", "TW": "Asia",
    "MO": "Asia", "MY": "Asia", "TH": "Asia", "VN": "Asia", "PH": "Asia", "ID": "Asia", "PK": "Asia",
    "BD": "Asia", "LK": "Asia", "NP": "Asia", "KH": "Asia", "LA": "Asia", "MM": "Asia", "MN": "Asia",
    "KZ": "Asia", "UZ": "Asia", "TM": "Asia", "KG": "Asia", "TJ": "Asia", "AZ": "Asia", "GE": "Asia",
    "AM": "Asia", "IL": "Asia", "SA": "Asia", "AE": "Asia", "QA": "Asia", "KW": "Asia", "BH": "Asia",
    "OM": "Asia", "JO": "Asia", "LB": "Asia", "IR": "Asia", "IQ": "Asia", "SY": "Asia", "YE": "Asia",
    "TR": "Asia", "PS": "Asia",
    # Africa
    "ZA": "Africa", "EG": "Africa", "NG": "Africa", "KE": "Africa", "ET": "Africa", "GH": "Africa",
    "TN": "Africa", "MA": "Africa", "DZ": "Africa", "UG": "Africa", "TZ": "Africa", "SN": "Africa",
    "CM": "Africa", "CI": "Africa", "ZW": "Africa", "ZM": "Africa", "RW": "Africa", "BW": "Africa",
    "MU": "Africa", "NA": "Africa", "MZ": "Africa", "AO": "Africa", "SD": "Africa", "LY": "Africa",
    # Oceania
    "AU": "Oceania", "NZ": "Oceania", "FJ": "Oceania", "PG": "Oceania", "NC": "Oceania",
}


# Country comes from OpenAlex (authoritative, global); region from the country map.
oa_country = (
    phase3_df["openalex_country_code"].fillna("UNK").astype(str).str.strip().str.upper().replace("", "UNK")
)
phase3_df["country_code"] = oa_country
phase3_df["region"] = oa_country.map(lambda cc: COUNTRY_TO_REGION.get(cc, "Unknown"))

# For QS Top50 matches, prefer the proposal-defined country/region (also fixes any
# rare cases where OpenAlex left the country blank). Keyed by normalized name so the
# curated OpenAlex variants resolve too.
top50_cr_by_key = {normalize_inst_key(name): cr for name, cr in TOP50_COUNTRY_REGION.items()}
for variant_key, canonical_full in QS_OPENALEX_VARIANTS.items():
    cr = TOP50_COUNTRY_REGION.get(canonical_full)
    if cr is not None:
        top50_cr_by_key[normalize_inst_key(variant_key)] = cr

top50_mask = phase3_df["is_top50_institution"]
if top50_mask.any():
    matched_keys = phase3_df.loc[top50_mask, "primary_institution_normalized"].map(normalize_inst_key)
    phase3_df.loc[top50_mask, "country_code"] = matched_keys.map(
        lambda k: top50_cr_by_key.get(k, ("UNK", "Unknown"))[0]
    )
    phase3_df.loc[top50_mask, "region"] = matched_keys.map(
        lambda k: top50_cr_by_key.get(k, ("UNK", "Unknown"))[1]
    )

# Labels: privileged (QS Top50) / underrepresented (other resolved institution) /
# unknown (no institution, ~0 because Phase 1 gates on institution presence).
inst_nonempty = phase3_df["primary_institution_normalized"].fillna("").str.strip().ne("")
phase3_df["privilege_label"] = "underrepresented"
phase3_df.loc[~inst_nonempty, "privilege_label"] = "unknown"
phase3_df.loc[phase3_df["is_top50_institution"], "privilege_label"] = "privileged"

print("Country/region mapping and fairness labels complete.")
print("Top50 matched:", int(phase3_df["is_top50_institution"].sum()))
print("Rows with non-UNK country:", int((phase3_df["country_code"] != "UNK").sum()))
print("Rows with known region:", int((phase3_df["region"] != "Unknown").sum()))
print("Label counts:")
print(phase3_df["privilege_label"].value_counts())

Country/region mapping and fairness labels complete.
Top50 matched: 8553
Rows with non-UNK country: 49849
Rows with known region: 49828
Label counts:
privilege_label
underrepresented    41447
privileged           8553
Name: count, dtype: int64


In [18]:
# --------------------------
# Export Phase 3 outputs + coverage summary
# --------------------------
phase3_output_columns = [
    "document_id",
    "document_text",
    "canonical_id",
    "id",
    "title",
    "abstract",
    "cleaned_title",
    "cleaned_abstract",
    "authors",
    "normalized_authors",
    "categories",
    "category_tokens",
    "primary_category",
    "update_date",
    "year",
    "primary_institution_raw",
    "primary_institution_normalized",
    "institution_match_method",
    "institution_match_key",
    "top50_canonical_institution",
    "country_code",
    "region",
    "is_top50_institution",
    "privilege_label",
]

phase3_optional_columns = [
    "openalex_primary_institution",
    "openalex_country_code",
    "institution_source",
    "institution_confidence_tier",
    "selection_rank",
]
phase3_output_columns.extend([c for c in phase3_optional_columns if c in phase3_df.columns])

phase3_output_df = phase3_df[phase3_output_columns].copy()

if len(phase3_output_df) != PHASE3_EXPECTED_ROWS:
    raise ValueError(f"Phase 3 output row count mismatch: {len(phase3_output_df)}")
if phase3_output_df["privilege_label"].isna().any():
    raise ValueError("privilege_label contains NA values")

duckdb.register("phase3_output_df_view", phase3_output_df)
duckdb.sql(f"COPY phase3_output_df_view TO '{phase3_output_path}' (FORMAT PARQUET);")
alias_map_df.to_csv(phase3_alias_map_path, index=False)

label_distribution_df = (
    phase3_output_df.groupby("privilege_label")
    .size()
    .reset_index(name="paper_count")
    .sort_values("paper_count", ascending=False)
)
label_distribution_df["ratio"] = label_distribution_df["paper_count"] / len(phase3_output_df)
label_distribution_df.to_csv(phase3_label_dist_path, index=False)

institution_coverage = float((phase3_output_df["primary_institution_normalized"].str.strip() != "").mean())
country_coverage = float((phase3_output_df["country_code"] != "UNK").mean())
region_coverage = float((phase3_output_df["region"] != "Unknown").mean())

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_path": phase3_input_path,
    "qs_top50_path": phase3_qs_path,
    "output_path": phase3_output_path,
    "row_count": len(phase3_output_df),
    "optional_columns_kept": [c for c in phase3_optional_columns if c in phase3_output_df.columns],
    "institution_coverage": institution_coverage,
    "country_coverage": country_coverage,
    "region_coverage": region_coverage,
    "top50_match_count": int(phase3_output_df["is_top50_institution"].sum()),
    "privilege_label_distribution": {
        row["privilege_label"]: int(row["paper_count"])
        for _, row in label_distribution_df.iterrows()
    },
    "top_matched_institutions": {
        k: int(v)
        for k, v in phase3_output_df["primary_institution_normalized"].value_counts().head(15).to_dict().items()
        if k
    },
}

with open(phase3_report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Phase 3 output written:", phase3_output_path)
print("Alias map written:", phase3_alias_map_path)
print("Label distribution written:", phase3_label_dist_path)
print("Match report written:", phase3_report_path)
print("\nCoverage summary:")
print("  institution_coverage:", round(institution_coverage, 4))
print("  country_coverage:", round(country_coverage, 4))
print("  region_coverage:", round(region_coverage, 4))
print("  top50_match_count:", int(phase3_output_df["is_top50_institution"].sum()))
print("\nLabel distribution:")
print(label_distribution_df)

Phase 3 output written: data\final_50k_labeled.parquet
Alias map written: data\institution_alias_map.csv
Label distribution written: data\label_distribution.csv
Match report written: data\affiliation_match_report.json

Coverage summary:
  institution_coverage: 1.0
  country_coverage: 0.997
  region_coverage: 0.9966
  top50_match_count: 8553

Label distribution:
    privilege_label  paper_count    ratio
1  underrepresented        41447  0.82894
0        privileged         8553  0.17106


## Phase 4. Corpus Statistics and Fairness Baseline Priors

- **Goal:** Produce report-ready corpus statistics and the corpus-level fairness priors used as the reference distribution for later retrieval-fairness analysis.
- **Tasks:** Summarize corpus distributions (year / category / institution / region / country), measure institution & country concentration (Top-k share, Gini, Lorenz), compute group proportions for `privilege_label` and `region`, and render figures.
- **Scope note:** Retrieval parity is intentionally deferred to after Phase 5. The baseline retriever (TF-IDF) and a curated CS topic query set will be compared against the dense retriever there; Phase 4 only establishes the corpus priors.
- **Expected outputs:** `final_corpus_statistics.csv`, `fairness_baseline_priors.json`, `corpus_summary.md`, and figures under `data/figures/`.

In [19]:
# --------------------------
# Phase 4 config + load labeled corpus
# --------------------------
import numpy as np
import matplotlib.pyplot as plt

phase4_input_path = os.path.join(DATA_DIR, "final_50k_labeled.parquet")
phase4_stats_path = os.path.join(DATA_DIR, "final_corpus_statistics.csv")
phase4_priors_path = os.path.join(DATA_DIR, "fairness_baseline_priors.json")
phase4_summary_path = os.path.join(DATA_DIR, "corpus_summary.md")
phase4_figures_dir = os.path.join(DATA_DIR, "figures")
os.makedirs(phase4_figures_dir, exist_ok=True)

PHASE4_EXPECTED_ROWS = TARGET_VALID_SIZE
PHASE4_REQUIRED_COLUMNS = [
    "document_id",
    "year",
    "primary_category",
    "region",
    "country_code",
    "privilege_label",
    "is_top50_institution",
    "primary_institution_normalized",
]

if not os.path.exists(phase4_input_path):
    raise FileNotFoundError(f"Phase 4 input not found: {phase4_input_path}")

phase4_df = duckdb.sql(f"SELECT * FROM read_parquet('{phase4_input_path}')").df()

if len(phase4_df) != PHASE4_EXPECTED_ROWS:
    raise ValueError(f"Unexpected row count: expected {PHASE4_EXPECTED_ROWS}, got {len(phase4_df)}")

missing_cols = [c for c in PHASE4_REQUIRED_COLUMNS if c not in phase4_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns for Phase 4: {missing_cols}")

# Normalize a few dtypes / empties so grouping is stable.
phase4_df["year"] = pd.to_numeric(phase4_df["year"], errors="coerce").astype("Int64")
phase4_df["privilege_label"] = phase4_df["privilege_label"].fillna("unknown").astype(str)
phase4_df["region"] = phase4_df["region"].fillna("Unknown").astype(str)
phase4_df["country_code"] = phase4_df["country_code"].fillna("UNK").astype(str)
phase4_df["primary_category"] = phase4_df["primary_category"].fillna("unknown").astype(str)
phase4_df["is_top50_institution"] = phase4_df["is_top50_institution"].fillna(False).astype(bool)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print("Phase 4 input:", phase4_input_path)
print("Rows:", len(phase4_df))
print("Figures dir:", phase4_figures_dir)

Phase 4 input: data\final_50k_labeled.parquet
Rows: 50000
Figures dir: data\figures


In [20]:
# --------------------------
# Descriptive corpus statistics (single-dim + concentration + cross-tabs)
# --------------------------
N = len(phase4_df)
stats_rows = []


def add_distribution(dimension, series, top_n=None):
    counts = series.value_counts(dropna=False)
    if top_n is not None:
        counts = counts.head(top_n)
    for group, cnt in counts.items():
        stats_rows.append(
            {"dimension": dimension, "group": str(group), "count": int(cnt), "ratio": float(cnt) / N}
        )


add_distribution("year", phase4_df["year"].astype("string"))
add_distribution("primary_category", phase4_df["primary_category"], top_n=20)
add_distribution("region", phase4_df["region"])
add_distribution("country_code", phase4_df["country_code"], top_n=25)
add_distribution("privilege_label", phase4_df["privilege_label"])
add_distribution(
    "is_top50_institution",
    phase4_df["is_top50_institution"].map({True: "top50", False: "non_top50"}),
)
if "institution_confidence_tier" in phase4_df.columns:
    add_distribution(
        "institution_confidence_tier",
        phase4_df["institution_confidence_tier"].fillna("unknown").astype(str),
    )
add_distribution(
    "institution_top25",
    phase4_df["primary_institution_normalized"].replace("", "(empty)"),
    top_n=25,
)


def gini(counts) -> float:
    arr = np.sort(np.asarray(counts, dtype=float))
    n = arr.size
    if n == 0 or arr.sum() == 0:
        return 0.0
    cum = np.cumsum(arr)
    return float((n + 1 - 2 * (cum / cum[-1]).sum()) / n)


inst_counts = phase4_df["primary_institution_normalized"].replace("", np.nan).dropna().value_counts()
country_counts = phase4_df.loc[phase4_df["country_code"] != "UNK", "country_code"].value_counts()

concentration = {
    "distinct_institutions": int(inst_counts.size),
    "distinct_countries": int(country_counts.size),
    "top10_institution_share": float(inst_counts.head(10).sum() / N),
    "top50_institution_share": float(inst_counts.head(50).sum() / N),
    "institution_gini": gini(inst_counts.values),
    "country_gini": gini(country_counts.values),
}
for k, v in concentration.items():
    stats_rows.append({"dimension": "concentration", "group": k, "count": np.nan, "ratio": float(v)})

corpus_statistics_df = pd.DataFrame(stats_rows, columns=["dimension", "group", "count", "ratio"])
corpus_statistics_df.to_csv(phase4_stats_path, index=False)

# Cross-tabs (kept in memory; reused by figures + report)
ct_label_region = pd.crosstab(phase4_df["privilege_label"], phase4_df["region"])
ct_label_category = pd.crosstab(phase4_df["privilege_label"], phase4_df["primary_category"])
ct_region_year = pd.crosstab(phase4_df["region"], phase4_df["year"])

print("Corpus statistics written:", phase4_stats_path)
print("\nprivilege_label distribution:")
print(phase4_df["privilege_label"].value_counts())
print("\nregion distribution:")
print(phase4_df["region"].value_counts())
print("\nConcentration:")
for k, v in concentration.items():
    print(f"  {k}: {round(v, 4) if isinstance(v, float) else v}")
print("\nprivilege_label x region:")
print(ct_label_region)

Corpus statistics written: data\final_corpus_statistics.csv

privilege_label distribution:
privilege_label
underrepresented    41447
privileged           8553
Name: count, dtype: int64

region distribution:
region
Europe           21528
North America    13507
Asia             12633
Oceania           1153
South America      745
Africa             262
Unknown            172
Name: count, dtype: int64

Concentration:
  distinct_institutions: 5937
  distinct_countries: 137
  top10_institution_share: 0.0643
  top50_institution_share: 0.2078
  institution_gini: 0.7525
  country_gini: 0.8654

privilege_label x region:
region            Africa   Asia  Europe  North America  Oceania  \
privilege_label                                                   
privileged             0   2174    2607           3539      233   
underrepresented     262  10459   18921           9968      920   

region            South America  Unknown  
privilege_label                           
privileged                 

In [21]:
# --------------------------
# Baseline group / fairness priors (corpus-level reference for later retrieval parity)
# --------------------------


def group_prior(series) -> dict:
    counts = series.value_counts(dropna=False)
    total = counts.sum()
    return {str(k): float(v) / total for k, v in counts.items()}


def representation_vs_uniform(prior: dict) -> dict:
    g = len(prior)
    uniform = 1.0 / g if g else 0.0
    return {k: (v / uniform if uniform else 0.0) for k, v in prior.items()}


privilege_prior = group_prior(phase4_df["privilege_label"])
region_prior = group_prior(phase4_df["region"])
privilege_repr_uniform = representation_vs_uniform(privilege_prior)
region_repr_uniform = representation_vs_uniform(region_prior)

p_priv = privilege_prior.get("privileged", 0.0)
p_under = privilege_prior.get("underrepresented", 0.0)
p_unknown = privilege_prior.get("unknown", 0.0)

baseline_priors = {
    "n_documents": int(len(phase4_df)),
    "privilege_prior": privilege_prior,
    "region_prior": region_prior,
    "privilege_representation_vs_uniform": privilege_repr_uniform,
    "region_representation_vs_uniform": region_repr_uniform,
    "privileged_to_underrepresented_ratio": (p_priv / p_under) if p_under else None,
    "institution_gini": concentration["institution_gini"],
    "country_gini": concentration["country_gini"],
    "top10_institution_share": concentration["top10_institution_share"],
    "top50_institution_share": concentration["top50_institution_share"],
}

with open(phase4_priors_path, "w", encoding="utf-8") as f:
    json.dump(baseline_priors, f, indent=2)

print("Baseline priors written:", phase4_priors_path)
print("\nprivilege_label prior:")
for k, v in privilege_prior.items():
    print(f"  {k}: {round(v, 4)}")
print("\nregion prior (top 8):")
for k, v in sorted(region_prior.items(), key=lambda x: x[1], reverse=True)[:8]:
    print(f"  {k}: {round(v, 4)}")
print(
    "\nprivileged : underrepresented =",
    round(p_priv, 4),
    ":",
    round(p_under, 4),
    "(ratio",
    round(p_priv / p_under, 3) if p_under else None,
    ")",
)

Baseline priors written: data\fairness_baseline_priors.json

privilege_label prior:
  underrepresented: 0.8289
  privileged: 0.1711

region prior (top 8):
  Europe: 0.4306
  North America: 0.2701
  Asia: 0.2527
  Oceania: 0.0231
  South America: 0.0149
  Africa: 0.0052
  Unknown: 0.0034

privileged : underrepresented = 0.1711 : 0.8289 (ratio 0.206 )


In [22]:
# --------------------------
# Figures
# --------------------------
saved_figures = []


def save_bar(series_counts, title, fname, rotate=0, top_n=None):
    data = series_counts if top_n is None else series_counts.head(top_n)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar([str(i) for i in data.index], data.values, color="#4C72B0")
    ax.set_title(title)
    ax.set_ylabel("papers")
    if rotate:
        plt.setp(ax.get_xticklabels(), rotation=rotate, ha="right")
    fig.tight_layout()
    path = os.path.join(phase4_figures_dir, fname)
    fig.savefig(path)
    plt.close(fig)
    saved_figures.append(path)


save_bar(phase4_df["privilege_label"].value_counts(), "Privilege label distribution", "privilege_label.png")
save_bar(phase4_df["region"].value_counts(), "Region distribution", "region.png", rotate=30)
save_bar(
    phase4_df["primary_category"].value_counts(),
    "Primary category (Top 15)",
    "primary_category.png",
    rotate=45,
    top_n=15,
)
save_bar(phase4_df["year"].value_counts().sort_index(), "Papers by year", "year.png")

# privilege_label x region heatmap
hm = ct_label_region.reindex(sorted(ct_label_region.columns), axis=1)
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(hm.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(hm.columns)))
ax.set_xticklabels(hm.columns, rotation=30, ha="right")
ax.set_yticks(range(len(hm.index)))
ax.set_yticklabels(hm.index)
ax.set_title("privilege_label x region (paper counts)")
fig.colorbar(im, ax=ax)
for i in range(len(hm.index)):
    for j in range(len(hm.columns)):
        ax.text(j, i, int(hm.values[i, j]), ha="center", va="center", fontsize=7, color="black")
fig.tight_layout()
_p = os.path.join(phase4_figures_dir, "privilege_by_region.png")
fig.savefig(_p)
plt.close(fig)
saved_figures.append(_p)

# Lorenz curve for institution concentration
inst_sorted = np.sort(inst_counts.values.astype(float))
cum_share = np.insert(np.cumsum(inst_sorted) / inst_sorted.sum(), 0, 0.0)
x = np.linspace(0, 1, len(cum_share))
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(x, cum_share, color="#C44E52", label=f"Lorenz (Gini={concentration['institution_gini']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", label="equality")
ax.set_title("Institution concentration (Lorenz)")
ax.set_xlabel("cumulative share of institutions")
ax.set_ylabel("cumulative share of papers")
ax.legend()
fig.tight_layout()
_p = os.path.join(phase4_figures_dir, "institution_lorenz.png")
fig.savefig(_p)
plt.close(fig)
saved_figures.append(_p)

print("Saved figures:")
for p in saved_figures:
    print(" ", p)

Saved figures:
  data\figures\privilege_label.png
  data\figures\region.png
  data\figures\primary_category.png
  data\figures\year.png
  data\figures\privilege_by_region.png
  data\figures\institution_lorenz.png


In [23]:
# --------------------------
# Narrative report (corpus_summary.md)
# --------------------------
top_categories = phase4_df["primary_category"].value_counts().head(10)
top_countries = phase4_df.loc[phase4_df["country_code"] != "UNK", "country_code"].value_counts().head(10)
top_institutions = inst_counts.head(10)
year_counts = phase4_df["year"].value_counts().sort_index()

lines = []
lines.append("# Corpus Summary (Phase 4)")
lines.append("")
lines.append(f"- Documents: {len(phase4_df)}")
lines.append(f"- Distinct institutions: {concentration['distinct_institutions']}")
lines.append(f"- Distinct countries: {concentration['distinct_countries']}")
lines.append(f"- Year range: {int(year_counts.index.min())}-{int(year_counts.index.max())}")
lines.append("")
lines.append("## Fairness group priors (privilege_label)")
for k in ["privileged", "underrepresented", "unknown"]:
    v = privilege_prior.get(k, 0.0)
    lines.append(f"- {k}: {v:.4f} ({int(round(v * len(phase4_df)))} papers)")
ratio = (p_priv / p_under) if p_under else float("nan")
lines.append(f"- privileged : underrepresented ratio: {ratio:.3f}")
lines.append("")
lines.append("## Region distribution")
for k, v in sorted(region_prior.items(), key=lambda x: x[1], reverse=True):
    lines.append(f"- {k}: {v:.4f}")
lines.append("")
lines.append("## Concentration / inequality")
lines.append(f"- Top-10 institution share: {concentration['top10_institution_share']:.4f}")
lines.append(f"- Top-50 institution share: {concentration['top50_institution_share']:.4f}")
lines.append(f"- Institution Gini: {concentration['institution_gini']:.4f}")
lines.append(f"- Country Gini: {concentration['country_gini']:.4f}")
lines.append("")
lines.append("## Top primary categories")
for k, v in top_categories.items():
    lines.append(f"- {k}: {int(v)}")
lines.append("")
lines.append("## Top countries")
for k, v in top_countries.items():
    lines.append(f"- {k}: {int(v)}")
lines.append("")
lines.append("## Top institutions")
for k, v in top_institutions.items():
    lines.append(f"- {k}: {int(v)}")
lines.append("")
lines.append("## Figures")
for p in saved_figures:
    lines.append(f"- {os.path.relpath(p, DATA_DIR)}")
lines.append("")
lines.append("## Note")
lines.append(
    "Retrieval fairness (TF-IDF baseline + curated CS topic queries, compared against the "
    "Phase 5 dense retriever) is deferred to after index construction. The priors above are "
    "the reference distribution for that analysis."
)

with open(phase4_summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Corpus summary written:", phase4_summary_path)
print()
print("\n".join(lines[:22]))

Corpus summary written: data\corpus_summary.md

# Corpus Summary (Phase 4)

- Documents: 50000
- Distinct institutions: 5937
- Distinct countries: 137
- Year range: 2024-2026

## Fairness group priors (privilege_label)
- privileged: 0.1711 (8553 papers)
- underrepresented: 0.8289 (41447 papers)
- unknown: 0.0000 (0 papers)
- privileged : underrepresented ratio: 0.206

## Region distribution
- Europe: 0.4306
- North America: 0.2701
- Asia: 0.2527
- Oceania: 0.0231
- South America: 0.0149
- Africa: 0.0052
- Unknown: 0.0034



## Phase 5. ChromaDB Index Construction and Retrieval Smoke Test

- **Goal:** Write the cleaned/labeled 50K corpus into ChromaDB and confirm the index is usable for later RAG experiments.
- **Tasks:** Select embedding model, ingest one-paper-one-document records with metadata into ChromaDB, verify collection count is exactly 50,000, run 5-10 smoke-test queries, and confirm top-k returns plus metadata/filter functionality.
- **Expected outputs:** `chroma_db/`, `chroma_ingestion_report.json`, `sample_retrieval_results.md`, `index_config.yaml`.